[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Lists


## What you will be able to do

Use a **list** to hold many values under one name, change them in place, and sort and search
them. You will also be able to explain why copying a list is not always copying, which is the
bug this notebook exists to prevent.


## The idea

### The problem

Every name so far has held exactly one value. That is enough until the moment you have more
than one thing, and you almost always have more than one thing.

The temperature at noon for a year. Every line in a file. All four hundred responses to a
survey. The results collected as a loop runs. You cannot invent four hundred names, and even if
you could, you could not write code that worked through them one at a time.

What you need is a way to hold many values under **one** name, keep them in a known order, and
reach any of them by position.

### What a list is

> A **list** is an ordered collection of values held under a single name. Square brackets
> create one and commas separate the items. Each item sits at a numbered position starting at
> `0`, exactly as characters do in a string.
>
> A list is **mutable**: it can be changed after it is created. Items can be replaced, added,
> removed and reordered, and the list itself stays the same list.

### Why mutability changes everything

The **Strings** notebook established that strings are immutable. Every operation that looked like changing
text actually built a new string. Lists work the opposite way, and that single difference is
why this notebook is the longest of the four container notebooks.

When a value can be changed in place, two questions appear that never arose before.

**Who else is looking at it?** If two names are attached to one list, a change made through
either is visible through both. The **Values and Variables** notebook previewed this and promised it would come back
properly. Here it is.

**What does copying mean?** Copying a list turns out to have two different answers depending on
what is inside it, and the answer that people assume is the one that causes the bug. The
manifest describes this notebook as covering "the copy that was not a copy", and that section
is the reason.

Neither question has a wrong answer in Python. Both have an answer that surprises you if you
have not been told.

### Where you will meet this

Lists arrive whether you ask for them or not. `"a,b,c".split(",")` returned one in the **Strings** notebook.
Reading a file gives a list of lines. A database query gives a list of rows. `re.findall` in
the **Regular Expressions** notebook gave you a list of matches. Almost anything producing several results produces a
list.

the **NumPy** guide replaces lists with NumPy arrays for numeric work, and the **Pandas** guide replaces them with
pandas columns for tables. Both are built on the ideas here.

### What this notebook covers

- Creating lists, and the positions and slices you already know from strings
- Changing an item in place, which a string could not do
- Adding, removing, searching, and the difference between `append` and `extend`
- Sorting two ways, and why one of them returns `None`
- Aliasing: two names on one list
- Copying, and the case where a copy is not a copy
- Changing a list while looping over it, which fails quietly
- Lists inside lists
- Four errors, including the one that turns your list into `None`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet:
read it, and read the output underneath it. Everything from Setup onward is where you start
running things, and the rest of the notebook takes this apart piece by piece.

```python
cities = ["Philadelphia", "Boston", "Denver"]

print(cities)
print(len(cities), "cities")
print(cities[0])
```

```
['Philadelphia', 'Boston', 'Denver']
3 cities
Philadelphia
```

`len` and `[0]` work exactly as they did for strings in the **Strings** notebook, and for the same reason:
both are sequences with positions, and positions start at 0.


## Setup

One import: `copy`, used once, in the section on copying, for `copy.deepcopy`. Everything else
in this notebook needs no library at all.


In [2]:
import copy

print("Ready.")


Ready.


## Worked examples

### Everything you learned about string positions still applies

Indexing, negative indexing and slicing are the same operations on a different container.


In [3]:
cities = ["Philadelphia", "Boston", "Denver", "Austin", "Seattle"]

print(cities[0])
print(cities[-1])
print(cities[1:3])      # from 1 up to but not including 3
print(cities[:2])
print(cities[::-1])


Philadelphia
Seattle
['Boston', 'Denver']
['Philadelphia', 'Boston']
['Seattle', 'Austin', 'Denver', 'Boston', 'Philadelphia']


A slice of a list gives a **new list**, just as a slice of a string gave a new string.

Lists can hold anything, including a mix of types, though in practice a list usually holds one
kind of thing.


In [4]:
mixed = [1, "two", 3.0, True, None]

print(mixed)
print(type(mixed[1]), type(mixed[2]))


[1, 'two', 3.0, True, None]
<class 'str'> <class 'float'>


### Here is what a string could not do

The **Strings** notebook tried `city[0] = "F"` and got a `TypeError`, because strings are immutable. Lists
are not.


In [5]:
cities = ["Philadelphia", "Boston", "Denver"]

cities[0] = "Pittsburgh"

print(cities)


['Pittsburgh', 'Boston', 'Denver']


No new list was made. The one that already existed changed. Everything difficult later in this
notebook follows from that.

### Adding items


In [6]:
cities = ["Philadelphia", "Boston"]

cities.append("Denver")          # one item, at the end
print(cities)

cities.insert(1, "Austin")       # one item, at a position
print(cities)

cities.extend(["Miami", "Reno"]) # several items
print(cities)


['Philadelphia', 'Boston', 'Denver']
['Philadelphia', 'Austin', 'Boston', 'Denver']
['Philadelphia', 'Austin', 'Boston', 'Denver', 'Miami', 'Reno']


`append` and `extend` are the pair people mix up. `append` adds **one item**, even when that
item is itself a list.


In [7]:
a = [1, 2]
b = [1, 2]

a.append([3, 4])     # one new item, which happens to be a list
b.extend([3, 4])     # two new items

print("append:", a)
print("extend:", b)


append: [1, 2, [3, 4]]
extend: [1, 2, 3, 4]


### Removing items

Four ways, and the difference is what you know at the time.


In [8]:
cities = ["Philadelphia", "Boston", "Denver", "Austin"]

cities.remove("Boston")     # by value
print(cities)

last = cities.pop()         # by position, and hands it back
print(last, "|", cities)

first = cities.pop(0)
print(first, "|", cities)


['Philadelphia', 'Denver', 'Austin']
Austin | ['Philadelphia', 'Denver']
Philadelphia | ['Denver']


`pop` returns the item it removed, which `remove` does not. Use `pop` when you want the value,
`remove` when you only want it gone.

### Searching


In [9]:
cities = ["Philadelphia", "Boston", "Denver", "Boston"]

print("Denver" in cities)
print(cities.index("Boston"))    # the first one
print(cities.count("Boston"))


True
1
2


`in` reads well and is the usual test. `index` gives the position of the first match and
raises if there is none, exactly like the string version in the **Strings** notebook.

### Numbers in a list


In [10]:
scores = [88, 72, 95, 64, 91]

print(len(scores), "scores")
print("total  ", sum(scores))
print("lowest ", min(scores))
print("highest", max(scores))
print("mean   ", sum(scores) / len(scores))


5 scores
total   410
lowest  64
highest 95
mean    82.0


### Turning a list back into text

`join` from the **Strings** notebook, now with its natural partner.


In [11]:
cities = ["Philadelphia", "Boston", "Denver"]

print(", ".join(cities))
print(" and ".join(cities))


Philadelphia, Boston, Denver
Philadelphia and Boston and Denver


### Sorting, and the trap that comes with it

There are two ways to sort, and they behave very differently.


In [12]:
scores = [88, 72, 95, 64]

print(sorted(scores))     # a new sorted list
print(scores)             # the original, untouched


[64, 72, 88, 95]
[88, 72, 95, 64]


In [13]:
scores.sort()             # sorts the list in place
print(scores)


[64, 72, 88, 95]


`sorted(x)` returns a new list. `x.sort()` changes `x` and returns **nothing**.

That "nothing" is a real `None`, and it causes one of the most common bugs in Python:


In [14]:
scores = [88, 72, 95, 64]

result = scores.sort()

print("result is", result)
print("but scores is", scores)


result is None
but scores is [64, 72, 88, 95]


The sort worked. The mistake is assigning its result to a name, because the result is `None`.
If you write `scores = scores.sort()` you destroy the list entirely, which Common errors below
shows.

The rule that covers this: **methods that change a list in place return `None`**. That is true
of `append`, `extend`, `insert`, `remove`, `sort` and `reverse`. When you want a value back,
use the function version instead: `sorted(x)`, or `reversed(x)`.

### Sorting by something other than the default


In [15]:
words = ["Zebra", "apple", "Mango"]

print(sorted(words))                  # capitals sort first, as in notebook 6
print(sorted(words, key=str.lower))   # the human answer
print(sorted(words, reverse=True))


['Mango', 'Zebra', 'apple']
['apple', 'Mango', 'Zebra']
['apple', 'Zebra', 'Mango']


`key` takes a rule for what to sort by, which is a function, and functions come in
the **Functions** notebook.
`str.lower` here means "sort by the lowercase version", without changing anything.


### Aliasing: two names on one list

The **Values and Variables** notebook previewed this with a list and promised it would come back. Here it is.

Assigning a list to another name does not copy it. Both names label the **same list**.


In [16]:
original = ["a", "b", "c"]
alias = original          # not a copy

alias.append("d")

print("original:", original)
print("alias:   ", alias)


original: ['a', 'b', 'c', 'd']
alias:    ['a', 'b', 'c', 'd']


Nothing in that code touched `original`, and `original` changed, because there is only one
list.

This is not a flaw. It is what makes lists useful to pass around. It is only a problem when you
believed you had a copy.

### Making an actual copy

Three ways, all equivalent.


In [17]:
original = ["a", "b", "c"]

copy_slice  = original[:]
copy_call   = list(original)
copy_method = original.copy()

original.append("d")

print("original:   ", original)
print("copy_slice: ", copy_slice)
print("copy_call:  ", copy_call)
print("copy_method:", copy_method)


original:    ['a', 'b', 'c', 'd']
copy_slice:  ['a', 'b', 'c']
copy_call:   ['a', 'b', 'c']
copy_method: ['a', 'b', 'c']


All three copies kept the three items. Use whichever reads best; `.copy()` says what it means.

### The copy that was not a copy

This one is easy to miss. Those three copies are **shallow**: they build
a new outer list, holding the same inner objects.

With a flat list of numbers or text that never matters. With a list of lists it matters a lot.


In [18]:
grid = [[1, 2], [3, 4]]
shallow = grid[:]          # a "copy"

shallow[0][0] = 99         # change something inside

print("grid:   ", grid)
print("shallow:", shallow)


grid:    [[99, 2], [3, 4]]
shallow: [[99, 2], [3, 4]]


Both changed. The outer list was copied, so `grid` and `shallow` are different lists, but the
inner lists were never copied and both outer lists point at the same two.

When the contents are themselves mutable, use `copy.deepcopy`, which copies all the way down.


In [19]:
grid = [[1, 2], [3, 4]]
deep = copy.deepcopy(grid)

deep[0][0] = 99

print("grid:", grid)
print("deep:", deep)


grid: [[1, 2], [3, 4]]
deep: [[99, 2], [3, 4]]


`deepcopy` is slower and you rarely need it. Reach for it when a copy is not behaving like a
copy, and the contents are lists, dictionaries or objects.


### The quiet mistake: changing a list while looping over it

This one does not raise. It silently gives a wrong answer.


In [20]:
numbers = [2, 4, 6]

for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)

print(numbers)


[4]


Every item is even, so the list should be empty. One survived.

The loop walks by position while the list is shrinking underneath it. After `2` is removed
everything shifts left, so the item now at position 1 is skipped entirely.

The fix is to never change the list you are looping over. Build a new one instead:


In [21]:
numbers = [2, 4, 6]

kept = [n for n in numbers if n % 2 != 0]

print(kept)


[]


That bracket form is a **comprehension**, which the **Comprehensions** notebook covers. A plain loop that appends to a
new list does the same job and is just as correct.


In [22]:
numbers = [2, 4, 6]
kept = []

for n in numbers:
    if n % 2 != 0:
        kept.append(n)

print(kept)


[]


### Lists inside lists

A list of lists is how you hold a grid or a table until you reach real tools for it.


In [23]:
grid = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

print(grid[1])         # the second row
print(grid[1][2])      # row 1, column 2
print(len(grid), "rows,", len(grid[0]), "columns")


[4, 5, 6]
6
3 rows, 3 columns


`grid[1][2]` reads left to right: take the row, then take from it. the **NumPy** guide replaces all of this
with NumPy arrays, which are better at it in every way, but the idea is the same.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/07-lists-solutions.ipynb).

**1.** Create a list `pets` holding `"dog"`, `"cat"` and `"fish"`. Print its length, then print
the last item without using the number 2.


In [24]:
# your code here


**2.** Add `"bird"` to the end of `pets`, insert `"hamster"` at the front, and print the
result.


In [25]:
# your code here


**3.** Given `scores = [88, 72, 95, 64, 91]`, print the highest, the lowest, and the mean
rounded to one decimal place.


In [26]:
# your code here


**4.** Given `words = ["delta", "Alpha", "charlie"]`, print them sorted so that capitals do not
come first, without changing `words` itself. Print `words` afterward to prove it is unchanged.


In [27]:
# your code here


**5.** Create a list, make a real copy of it, change the copy, and print both to show the
original did not move.


In [28]:
# your code here


**6.** Given `numbers = [1, 2, 3, 4, 5, 6]`, build a **new** list holding only the even ones,
without changing `numbers`. A loop is fine.


In [29]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### IndexError: list index out of range

The same error strings gave in the **Strings** notebook, for the same reason.


In [30]:
pets = ["dog", "cat", "fish"]
print(pets[3])


IndexError: list index out of range

Three items means valid positions `0`, `1` and `2`. `pets[3]` is one past the end, and the
count-from-one habit is what puts it there.

`pets[-1]` is the safe way to say "the last one".


### TypeError: the list became None

This is the sorting trap made real, and it is worth seeing fail.


In [31]:
scores = [88, 72, 95]
scores = scores.sort()      # sort() returns None, so scores is now None

print(scores[0])


TypeError: 'NoneType' object is not subscriptable

`'NoneType' object is not subscriptable` means `scores` is no longer a list. The list was
sorted correctly and then thrown away by the assignment.

Either sort in place and do not assign, or use `sorted` and do:


In [32]:
scores = [88, 72, 95]
scores.sort()               # in place, no assignment
print(scores)

scores = [88, 72, 95]
scores = sorted(scores)     # a new list, assigned
print(scores)


[72, 88, 95]
[72, 88, 95]


### ValueError: remove needs something that is there

`remove` raises when the value is absent.


In [33]:
pets = ["dog", "cat"]
pets.remove("iguana")


ValueError: list.remove(x): x not in list

`list.remove(x): x not in list` is the same situation `index` reports in the **Strings** notebook. Check
first when a miss is possible:

```
if "iguana" in pets:
    pets.remove("iguana")
```


### TypeError: adding a bare value to a list

`+` joins two lists. It does not add an item.


In [34]:
print([1, 2] + 3)


TypeError: can only concatenate list (not "int") to list

`can only concatenate list (not "int") to list` means the right side has to be a list too.
Either wrap it or use `append`.


In [35]:
print([1, 2] + [3])

nums = [1, 2]
nums.append(3)
print(nums)


[1, 2, 3]
[1, 2, 3]


## Recap

- A list holds many values in order, and positions work exactly as they do for strings.
- Lists are **mutable**: `items[0] = x`, `append`, `remove` all change the list itself.
- Methods that change a list in place return `None`, so never write `x = x.sort()`.
- `sorted(x)` returns a new list; `x.sort()` changes `x` and returns nothing.
- Assigning a list to a second name gives an alias, not a copy.
- `x[:]`, `list(x)` and `x.copy()` copy the outer list only; nested contents are still shared.
- `copy.deepcopy(x)` copies all the way down, for when the contents are mutable too.
- Never add to or remove from a list while looping over it. Build a new list.


## What is next

The **Tuples and Unpacking** notebook, which is the same idea with one thing removed: a tuple
cannot be changed. That restriction turns out to be a feature, and it is what makes
`a, b = b, a` from the **Values and Variables** notebook work.


---

&#8592; **Previous:** [Booleans and Comparison](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/06-booleans-and-comparison.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)  &nbsp;·&nbsp;  **Next:** [Tuples and Unpacking](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/08-tuples-and-unpacking.ipynb) &#8594;
